In [8]:
# Cell 1 — Config
from pathlib import Path
import pandas as pd, json, re, cv2
from collections import Counter, defaultdict

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST = ROOT / "data" / "ocr_frames" / "manifest.csv"
OUT      = ROOT / "data" / "ocr_ws6"
OUT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST)
CAM_WEIGHT = {"back_right": 3, "front": 2, "back_left": 1}

print("crops:", len(manifest))
print("per camera:", manifest.groupby("camera").size().to_dict())
print("per colour:", manifest.groupby("color").size().to_dict())
print("bottles:", manifest.groupby(["video","Sequence"]).size().shape[0])
manifest.head()

crops: 524
per camera: {'back_left': 179, 'back_right': 179, 'front': 166}
per colour: {'black': 443, 'cyan': 81}
bottles: 182


,video,Sequence,t_sec,camera,priority,crop_path,gt_tare,gt_year,color
0,14_43,542436,39.0,front,1,data\ocr_frames\crop_14_43_542436_front.jpg,10.8,2033,black
1,14_43,542436,39.0,back_left,2,data\ocr_frames\crop_14_43_542436_back_left.jpg,10.8,2033,black
2,14_43,542436,39.0,back_right,0,data\ocr_frames\crop_14_43_542436_back_right.jpg,10.8,2033,black
3,14_43,542437,43.0,front,1,data\ocr_frames\crop_14_43_542437_front.jpg,10.7,2036,black
4,14_43,542437,43.0,back_left,2,data\ocr_frames\crop_14_43_542437_back_left.jpg,10.7,2036,black


In [9]:
# Cell 2 — preprocess (kept mild; metal text dislikes heavy blur)
def preprocess(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
    g = cv2.bilateralFilter(g, 9, 75, 75)
    g = cv2.resize(g, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    return g

In [13]:
import time, cv2, easyocr

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

sample = manifest.sample(8, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    h, w = img.shape[:2]

    t0 = time.time()
    reader.readtext(img, detail=0)
    t_orig = time.time() - t0

    img_small = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img_small, detail=0, canvas_size=736, mag_ratio=1.0)
    t_fast = time.time() - t0

    print(f"{r.camera:10s} {r.Sequence:6d}  size={h}x{w}  orig={t_orig*1000:.0f} ms  fast={t_fast*1000:.0f} ms")

front          59  size=392x301  orig=1452 ms  fast=1447 ms
back_left  542442  size=318x369  orig=1354 ms  fast=1320 ms
back_left      41  size=381x556  orig=2560 ms  fast=2499 ms
back_left  542510  size=396x441  orig=1990 ms  fast=2251 ms
front          21  size=364x459  orig=2436 ms  fast=2073 ms
back_right 542586  size=396x586  orig=2751 ms  fast=2752 ms
front      542498  size=168x410  orig=880 ms  fast=849 ms
front          11  size=368x252  orig=1140 ms  fast=1127 ms


In [15]:
import torch, os, time, cv2, easyocr

# 1. use all CPU cores
torch.set_num_threads(os.cpu_count())
torch.set_num_interop_threads(1)
print("torch threads:", torch.get_num_threads(), "| cpu count:", os.cpu_count())

# 2. helper
def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

# 3. reader
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

# 4. quick benchmark on 5 crops
sample = manifest.sample(5, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    img = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img, detail=0, canvas_size=736, mag_ratio=1.0,
                    batch_size=1, decoder="greedy", paragraph=False)
    print(f"{r.camera:10s} {r.Sequence:>8}  size={img.shape[0]}x{img.shape[1]}  {(time.time()-t0)*1000:.0f} ms")

torch threads: 16 | cpu count: 16
front            59  size=392x301  430 ms
back_left    542442  size=318x369  404 ms
back_left        41  size=381x556  751 ms
back_left    542510  size=396x441  460 ms
front            21  size=364x459  605 ms


In [16]:
# Cell 3 — cap_size defined first, then OCR loop
import easyocr, time

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

TARE_RE = re.compile(r'(\d{1,2})[,.](\d)\s*[Kk][Gg]')
YEAR_RE = re.compile(r'\b(19|20)\d{2}\b')
STICKER_KG = {12.5, 10.5}

def parse(txts):
    joined = " ".join(txts).replace(" ", "")
    tare = None
    m = TARE_RE.search(joined)
    if m:
        v = float(f"{m.group(1)}.{m.group(2)}")
        if v not in STICKER_KG and 5.0 <= v <= 25.0:
            tare = v
        else:
            for tok in re.findall(r'\d{1,2}[,.]\d', joined):
                vv = float(tok.replace(",", "."))
                if vv not in STICKER_KG and 5.0 <= vv <= 25.0:
                    tare = vv; break
    year = None
    for tok in txts:
        y = YEAR_RE.search(tok)
        if y and 1990 <= int(y.group(0)) <= 2035:
            year = int(y.group(0)); break
    return tare, year

manifest = manifest.drop_duplicates(subset=["video", "Sequence", "camera"]).reset_index(drop=True)
print(f"manifest: {len(manifest)} crops")

crop_rows = []
t_start = time.time()

for i, r in enumerate(manifest.itertuples(index=False), 1):
    img = cv2.imread(str(ROOT / r.crop_path))
    if img is None:
        continue
    img = cap_size(img, 640)
    txts = reader.readtext(
        img, detail=0,
        canvas_size=736,
        mag_ratio=1.0,
        batch_size=8,
        decoder="greedy",
    )
    tare, year = parse(txts)
    crop_rows.append({
        "video": r.video, "Sequence": int(r.Sequence),
        "camera": r.camera, "priority": int(r.priority),
        "color": r.color,
        "gt_tare": r.gt_tare, "gt_year": r.gt_year,
        "raw": str(txts),
        "tare": tare, "year": year,
    })
    if i % 40 == 0:
        elapsed = time.time() - t_start
        per_crop = elapsed / i
        remaining = per_crop * (len(manifest) - i)
        print(f"  {i}/{len(manifest)}  ({elapsed:.0f}s elapsed, ~{remaining:.0f}s left, {per_crop*1000:.0f} ms/crop)")

crop_df = pd.DataFrame(crop_rows)
crop_df.to_csv(OUT / "ocr_per_crop.csv", index=False)
print(f"\ncrops OCR'd: {len(crop_df)}",
      f"| crops with tare: {int(crop_df.tare.notna().sum())}",
      f"| crops with year: {int(crop_df.year.notna().sum())}",
      f"| total time: {time.time()-t_start:.0f}s")

manifest: 524 crops
  40/524  (23s elapsed, ~274s left, 566 ms/crop)
  80/524  (43s elapsed, ~240s left, 540 ms/crop)
  120/524  (64s elapsed, ~215s left, 531 ms/crop)
  160/524  (85s elapsed, ~192s left, 529 ms/crop)
  200/524  (109s elapsed, ~176s left, 544 ms/crop)
  240/524  (133s elapsed, ~158s left, 555 ms/crop)
  280/524  (158s elapsed, ~137s left, 563 ms/crop)
  320/524  (178s elapsed, ~113s left, 556 ms/crop)
  360/524  (200s elapsed, ~91s left, 555 ms/crop)
  400/524  (220s elapsed, ~68s left, 551 ms/crop)
  440/524  (242s elapsed, ~46s left, 551 ms/crop)
  480/524  (264s elapsed, ~24s left, 550 ms/crop)
  520/524  (286s elapsed, ~2s left, 550 ms/crop)

crops OCR'd: 524 | crops with tare: 2 | crops with year: 7 | total time: 288s


In [19]:
# Cell 4 — weighted majority vote per bottle, per-camera and per-colour metrics
def weighted_vote(rows, field):
    scores = defaultdict(float)
    for r in rows:
        v = r.get(field)
        if v is None or pd.isna(v):
            continue
        scores[v] += CAM_WEIGHT.get(r.get("camera"), 1)
    if not scores:
        return None
    return max(scores.items(), key=lambda kv: kv[1])[0]

def plain_vote(rows, field):
    vals = [r[field] for r in rows if r.get(field) is not None and not pd.isna(r.get(field))]
    return Counter(vals).most_common(1)[0][0] if vals else None

per_bottle = []
for (vk, seq), grp in crop_df.groupby(["video", "Sequence"]):
    rows = grp.to_dict("records")
    per_bottle.append({
        "video": vk, "Sequence": seq,
        "color": rows[0]["color"],
        "n_cameras_attempted": len(rows),
        "cameras": "+".join(sorted(r["camera"] for r in rows)),
        "tare":       weighted_vote(rows, "tare"),
        "year":       weighted_vote(rows, "year"),
        "tare_plain": plain_vote(rows, "tare"),
        "year_plain": plain_vote(rows, "year"),
        "gt_tare":    rows[0]["gt_tare"],
        "gt_year":    rows[0]["gt_year"],
    })

bottle_df = pd.DataFrame(per_bottle)
bottle_df["tare_ok"]       = (bottle_df.tare       == bottle_df.gt_tare).astype(int)
bottle_df["year_ok"]       = (bottle_df.year       == bottle_df.gt_year).astype(int)
bottle_df["tare_ok_plain"] = (bottle_df.tare_plain == bottle_df.gt_tare).astype(int)
bottle_df["year_ok_plain"] = (bottle_df.year_plain == bottle_df.gt_year).astype(int)

# overall
n = len(bottle_df)
overall = {
    "n_bottles":               n,
    "tare_exact_match_weighted": round(int(bottle_df.tare_ok.sum()) / n, 3),
    "year_exact_match_weighted": round(int(bottle_df.year_ok.sum()) / n, 3),
    "both_exact_match_weighted": round(int(((bottle_df.tare_ok == 1) & (bottle_df.year_ok == 1)).sum()) / n, 3),
    "tare_exact_match_plain":    round(int(bottle_df.tare_ok_plain.sum()) / n, 3),
    "year_exact_match_plain":    round(int(bottle_df.year_ok_plain.sum()) / n, 3),
}

# per-colour
per_color = {}
for color, grp in bottle_df.groupby("color"):
    m = len(grp)
    per_color[color] = {
        "bottles": m,
        "tare_exact_match_weighted": round(int(grp.tare_ok.sum()) / m, 3),
        "year_exact_match_weighted": round(int(grp.year_ok.sum()) / m, 3),
        "both_exact_match_weighted": round(int(((grp.tare_ok == 1) & (grp.year_ok == 1)).sum()) / m, 3),
    }

# back-right alone
br = crop_df[crop_df.camera == "back_right"].copy()
br = br.drop_duplicates(subset=["video", "Sequence"]).reset_index(drop=True)

# diagnostics: guard against every type of silent surprise
print("br rows:", len(br))
print("br columns:", list(br.columns))

br_metrics = {
    "back_right_crops":         int(len(br)),
    "back_right_tare_hits":     int(br["tare"].notna().sum()),
    "back_right_year_hits":     int(br["year"].notna().sum()),
    "back_right_tare_correct":  int((br["tare"] == br["gt_tare"]).sum()),
    "back_right_year_correct":  int((br["year"] == br["gt_year"]).sum()),
}
print("br_metrics:", br_metrics)

result = {
    "overall": overall,
    "per_color": per_color,
    "back_right_only": br_metrics,
    "coverage": {
        "gt_bottles_targeted": int(len(manifest.groupby(['video','Sequence']).size())),
        "bottles_with_3_cameras": int((bottle_df.n_cameras_attempted == 3).sum()),
        "bottles_with_2_cameras": int((bottle_df.n_cameras_attempted == 2).sum()),
        "bottles_with_1_camera":  int((bottle_df.n_cameras_attempted == 1).sum()),
        "total_crops":            int(len(crop_df)),
    },
}
print(json.dumps(result, indent=2))
(OUT / "ocr_metrics.json").write_text(json.dumps(result, indent=2))
bottle_df.to_csv(OUT / "ocr_per_bottle.csv", index=False)
print("\nsaved:", (OUT / "ocr_metrics.json").relative_to(ROOT))

br rows: 179
br columns: ['video', 'Sequence', 'camera', 'priority', 'color', 'gt_tare', 'gt_year', 'raw', 'tare', 'year']
br_metrics: {'back_right_crops': 179, 'back_right_tare_hits': 0, 'back_right_year_hits': 5, 'back_right_tare_correct': 0, 'back_right_year_correct': 0}
{
  "overall": {
    "n_bottles": 182,
    "tare_exact_match_weighted": 0.005,
    "year_exact_match_weighted": 0.0,
    "both_exact_match_weighted": 0.0,
    "tare_exact_match_plain": 0.005,
    "year_exact_match_plain": 0.0
  },
  "per_color": {
    "black": {
      "bottles": 154,
      "tare_exact_match_weighted": 0.006,
      "year_exact_match_weighted": 0.0,
      "both_exact_match_weighted": 0.0
    },
    "cyan": {
      "bottles": 28,
      "tare_exact_match_weighted": 0.0,
      "year_exact_match_weighted": 0.0,
      "both_exact_match_weighted": 0.0
    }
  },
  "back_right_only": {
    "back_right_crops": 179,
    "back_right_tare_hits": 0,
    "back_right_year_hits": 5,
    "back_right_tare_correct": 0,

In [20]:
# Cell 5 — sanity check
print("crops:", len(crop_df),
      "| with tare:", int(crop_df.tare.notna().sum()),
      "| with year:", int(crop_df.year.notna().sum()))
print("\n5 random raw-text examples:")
for t in crop_df.sample(min(5, len(crop_df)), random_state=0)["raw"].tolist():
    print("  ", t[:140])
print("\n5 random crops where tare WAS parsed:")
parsed = crop_df[crop_df.tare.notna()]
if len(parsed) > 0:
    for _, r in parsed.sample(min(5, len(parsed)), random_state=0).iterrows():
        print(f"  {r.video} seq={r.Sequence} cam={r.camera} tare={r.tare} gt={r.gt_tare}  raw={r.raw[:80]}")
else:
    print("  (none)")

crops: 524 | with tare: 2 | with year: 7

5 random raw-text examples:
   []
   []
   ['K6.2n3s']
   []
   ['92']

5 random crops where tare WAS parsed:
  14_43 seq=542558 cam=front tare=10.8 gt=15.7  raw=['P', '903', "'10.8", 'KG-2037', 'T3t-']
  14_43 seq=542440 cam=front tare=10.6 gt=10.6  raw=['7', '10.6 KG', '3', 'UN1965', 'PISV:']
